# Loading and Saving Data with PySpark

## Installing dependencies

In [ ]:
%pip install pyspark pandas dotenv

import pyspark as py

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"


## Create SparkSession & DataFrames

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("olist-bronze") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

In [ ]:
df_orders = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("mode", "PERMISSIVE") \
    .csv("../raw/olist_orders_dataset.csv")

df_products = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("mode", "PERMISSIVE") \
    .csv("../raw/olist_products_dataset.csv")

df_customers = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("mode", "PERMISSIVE") \
    .csv("../raw/olist_customers_dataset.csv")

df_order_items = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("mode", "PERMISSIVE") \
    .csv("../raw/olist_order_items_dataset.csv")

df_order_payments = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("mode", "PERMISSIVE") \
    .csv("../raw/olist_order_payments_dataset.csv")

df_geolocation = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("mode", "PERMISSIVE") \
    .csv("../raw/olist_geolocation_dataset.csv")

df_reviews = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("mode", "PERMISSIVE") \
    .csv("../raw/olist_order_reviews_dataset.csv")

df_sellers = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("mode", "PERMISSIVE") \
    .csv("../raw/olist_sellers_dataset.csv")

df_product_category_name_translation = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("mode", "PERMISSIVE") \
    .csv("../raw/product_category_name_translation.csv")

In [ ]:
df_reviews.printSchema()       # types des colonnes
df_reviews.show(5)             # aperçu des données
df_reviews.count()             # nombre de lignes
df_reviews.describe().show()   # statistiques descriptives

In [ ]:
from pyspark.sql.functions import col, sum as _sum

null_counts = df_orders.select([
    _sum(col(c).isNull().cast("int")).alias(c)
    for c in df_orders.columns
])
null_counts.show()

## Identifier Clés de Jointures

In [ ]:
for name, df in {
    "orders": df_orders,
    "products": df_products,
    "customers": df_customers,
    "order_items": df_order_items,
    "order_payments": df_order_payments,
    "geolocation": df_geolocation,
    "order_reviews": df_reviews,
    "sellers": df_sellers,
    "category_translation": df_product_category_name_translation,
}.items():
    print(f"\n=== {name} ===")
    print(df.columns)

In [ ]:
from pyspark.sql.functions import countDistinct

# order_id présent dans orders, order_items, payments, reviews ?
print("order_id distincts dans orders :", df_orders.select(countDistinct("order_id")).collect()[0][0])
print("order_id distincts dans order_items :", df_order_items.select(countDistinct("order_id")).collect()[0][0])
print("order_id distincts dans payments :", df_order_payments.select(countDistinct("order_id")).collect()[0][0])
print("order_id distincts dans reviews :", df_reviews.select(countDistinct("order_id")).collect()[0][0])

# product_id présent dans order_items et products ?
print("\nproduct_id distincts dans order_items :", df_order_items.select(countDistinct("product_id")).collect()[0][0])
print("product_id distincts dans products :", df_products.select(countDistinct("product_id")).collect()[0][0])

# seller_id présent dans order_items et sellers ?
print("\nseller_id distincts dans order_items :", df_order_items.select(countDistinct("seller_id")).collect()[0][0])
print("seller_id distincts dans sellers :", df_sellers.select(countDistinct("seller_id")).collect()[0][0])

# customer_id présent dans orders et customers ?
print("\ncustomer_id distincts dans orders :", df_orders.select(countDistinct("customer_id")).collect()[0][0])
print("customer_id distincts dans customers :", df_customers.select(countDistinct("customer_id")).collect()[0][0])

In [ ]:
# Y a-t-il des order_id dans order_items sans correspondance dans orders ?
orphan_items = df_order_items.join(df_orders, "order_id", "left_anti")
print("Articles sans commande correspondante :", orphan_items.count())

# Y a-t-il des product_id dans order_items sans correspondance dans products ?
orphan_products = df_order_items.join(df_products, "product_id", "left_anti")
print("Produits dans order_items absents de products :", orphan_products.count())

# Y a-t-il des seller_id dans order_items sans correspondance dans sellers ?
orphan_sellers = df_order_items.join(df_sellers, "seller_id", "left_anti")
print("Vendeurs dans order_items absents de sellers :", orphan_sellers.count())

## Save as Parquet

In [ ]:
df_orders.write \
    .mode("overwrite") \
    .parquet("../data/bronze/orders/")

df_products.write \
    .mode("overwrite") \
    .parquet("../data/bronze/products/")

df_customers.write \
    .mode("overwrite") \
    .parquet("../data/bronze/customers/")

df_order_items.write \
    .mode("overwrite") \
    .parquet("../data/bronze/order_items/")

df_order_payments.write \
    .mode("overwrite") \
    .parquet("../data/bronze/order_payments/")

df_geolocation.write \
    .mode("overwrite") \
    .parquet("../data/bronze/geolocation/")

df_reviews.write \
    .mode("overwrite") \
    .parquet("../data/bronze/order_reviews/")

df_sellers.write \
    .mode("overwrite") \
    .parquet("../data/bronze/sellers/")

df_product_category_name_translation.write \
    .mode("overwrite") \
    .parquet("../data/bronze/product_category_name_translation/")